In [ ]:
# Train a from-scratch MLP on MNIST using the scalar Value autograd engine.
# Build Neuron -> Layer -> MLP on top of micrograd_value.Value, then train with softmax cross-entropy + mini-batch SGD.
#
# Value is a scaler engine (one graph node per multiply/add), so full MNIST is impractical here. To keep it runnable: 4x4 average-pool each image 28x28 -> 7x7
# (784 -> 49 inputs), train on a small subset, small network, few epochs.

In [ ]:
from __future__ import annotations

import random

import numpy as np
from torchvision import datasets

from micrograd_value import Value

In [ ]:
# Network: Neuron -> Layer -> MLP, each a Module exposing parameters()/zero_grad().

class Module:
    """Base class providing ``parameters()`` and ``zero_grad()``."""

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0

    def parameters(self):
        return []


class Neuron(Module):
    def __init__(self, n_in, nonlin=True):
        scale = n_in ** -0.5
        self.w = [Value(random.uniform(-1, 1) * scale) for _ in range(n_in)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.relu() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]


class Layer(Module):
    def __init__(self, n_in, n_out, nonlin=True):
        self.neurons = [Neuron(n_in, nonlin=nonlin) for _ in range(n_out)]

    def __call__(self, x):
        return [n(x) for n in self.neurons]

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP(Module):

    def __init__(self, sizes):
        self.layers = []
        for i in range(len(sizes) - 1):
            last = i == len(sizes) - 2
            self.layers.append(Layer(sizes[i], sizes[i + 1], nonlin=not last))

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [ ]:
# Loss: softmax cross-entropy over a list of logit Values, for one example.
# Numerically stabilized via log-sum-exp; gradient collapses to softmax - onehot.

def softmax_cross_entropy(logits, target):
    max_logit = max(l.data for l in logits)
    exps = [(l - max_logit).exp() for l in logits]
    denom = sum(exps, Value(0.0))
    log_prob = (exps[target] / denom).log()
    return -log_prob

In [ ]:
# Data: load MNIST, average-pool 28x28 -> 7x7, normalize to [0, 1], subsample.

def avg_pool(img, k=4):
    s = 28 // k
    x = np.asarray(img, dtype=np.float32) / 255.0
    x = x.reshape(s, k, s, k).mean(axis=(1, 3))
    return x.reshape(-1).tolist()


def load_mnist(n_train, n_test, pool=4, seed=0):
    train = datasets.MNIST(root="./data", train=True, download=True)
    test = datasets.MNIST(root="./data", train=False, download=True)

    rng = random.Random(seed)
    tr_idx = rng.sample(range(len(train)), n_train)
    te_idx = rng.sample(range(len(test)), n_test)

    Xtr = [avg_pool(train[i][0], pool) for i in tr_idx]
    ytr = [train[i][1] for i in tr_idx]
    Xte = [avg_pool(test[i][0], pool) for i in te_idx]
    yte = [test[i][1] for i in te_idx]
    return Xtr, ytr, Xte, yte

In [ ]:
# Evaluation: fraction of examples whose argmax logit matches the label.

def accuracy(model, X, y):
    correct = 0
    for xi, yi in zip(X, y):
        logits = model([Value(v) for v in xi])
        pred = max(range(len(logits)), key=lambda k: logits[k].data)
        correct += int(pred == yi)
    return correct / len(X)

In [ ]:
# Training loop: mini-batch SGD over the subset, reporting loss and accuracy.

def train():
    random.seed(1337)

    N_TRAIN, N_TEST = 300, 200
    POOL = 4
    EPOCHS = 15
    BATCH = 16
    LR = 0.2

    print("Loading MNIST (downsampling 28x28 -> 7x7)...")
    Xtr, ytr, Xte, yte = load_mnist(N_TRAIN, N_TEST, pool=POOL)
    n_in = len(Xtr[0])

    model = MLP([n_in, 32, 10])
    n_params = len(model.parameters())
    print(f"MLP [{n_in}, 32, 10] with {n_params} parameters")
    print(f"train={N_TRAIN}  test={N_TEST}  batch={BATCH}  lr={LR}\n")

    idx = list(range(N_TRAIN))
    for epoch in range(1, EPOCHS + 1):
        random.shuffle(idx)
        epoch_loss = 0.0
        n_batches = 0

        for start in range(0, N_TRAIN, BATCH):
            batch = idx[start:start + BATCH]

            losses = []
            for i in batch:
                logits = model([Value(v) for v in Xtr[i]])
                losses.append(softmax_cross_entropy(logits, ytr[i]))
            loss = sum(losses, Value(0.0)) / len(batch)

            model.zero_grad()
            loss.backward()

            for p in model.parameters():
                p.data -= LR * p.grad

            epoch_loss += loss.data
            n_batches += 1

        train_acc = accuracy(model, Xtr, ytr)
        test_acc = accuracy(model, Xte, yte)
        print(f"epoch {epoch:2d} | loss {epoch_loss / n_batches:.4f} "
              f"| train acc {train_acc:.3f} | test acc {test_acc:.3f}")

    print("\nDone. (Subset + scalar engine - accuracy is illustrative only.)")

In [ ]:
train()